# gprMax B-scan forward modelling results
In this notebook you can:
- Visualise baseline and timelapse B-scans (time vs Tx position heatmap)
- Wiggle / seismic-section plots
- Timelapse difference (×DIFF_GAIN) for both representations
- Wavefield snapshots at the domain-centre trace

In [ ]:
import os
import glob
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as patches
import h5py
import pyvista as pv

matplotlib.rcParams.update({"figure.dpi": 100})

# ── File paths & run configuration ───────────────────────────────────────────
NOTEBOOK_DIR  = os.path.dirname(os.path.abspath("__file__"))
file_prefix   = ""     # match file_prefix in BScan_generator; "" = default names
output_subdir = "Experiment_4"     # e.g. "bscan_01" -> outputs/bscan_01/;  "" = same dir as .in files

out_dir = (
    os.path.join(NOTEBOOK_DIR, "", output_subdir)
    if output_subdir else NOTEBOOK_DIR
)

def _stem(suffix):
    return f"{file_prefix}_{suffix}" if file_prefix else suffix

BASELINE_OUT  = os.path.join(out_dir, _stem("baseline")  + "_merged.out")
TIMELAPSE_OUT = os.path.join(out_dir, _stem("timelapse") + "_merged.out")

# ── Geometry (copy from BScan_generator Cell 1) ───────────────────────────────
c_light        = 3e8           # [m/s]
eps_ice        = 3.15
f_central      = 1.5e9         # [Hz]
domain_width   = 4.0           # [m]
air_thickness  = 0.1           # [m]
fracture_depth = 0.6           # [m]

c_ice = c_light / np.sqrt(eps_ice)

# ── B-scan parameters (copy printed values from BScan_generator Cell 2) ──────
tx_start_x    = 0.001491   # [m]  scan_margin (= 2*dx)
trace_spacing = 0.011269   # [m]  trace spacing snapped to grid
n_traces      = 356        # ★ Total traces from Cell 2 output
offset        = 0.02       # [m]  source-receiver separation
center_trace  = 180        # 1-indexed — "Snapshot at trace" from Cell 2 output

# ── Derived scan positions ────────────────────────────────────────────────────
tx_positions = tx_start_x + np.arange(n_traces) * trace_spacing
center_tx_x  = tx_start_x + (center_trace - 1) * trace_spacing
center_rx_x  = center_tx_x + offset

# ── Snapshot directories (gprMax creates _snaps{run_number}/ per trace) ──────
BASELINE_SNAPS_DIR  = os.path.join(out_dir, _stem("baseline")  + f"_snaps{center_trace}")
TIMELAPSE_SNAPS_DIR = os.path.join(out_dir, _stem("timelapse") + f"_snaps{center_trace}")

# ── Difference display controls ───────────────────────────────────────────────
DIFF_GAIN    = 20.0   # multiply residual to reveal subwavelength scattering
SNAP_INDICES = None   # None = all;  e.g. [5, 10, 15, 20, 25] for a subset
snap_interval = 0.5e-9  # [s] must match snap_interval in BScan_generator Cell 1

print(f"Baseline  .out : {BASELINE_OUT}")
print(f"Timelapse .out : {TIMELAPSE_OUT}")
print(f"Snaps (B)      : {BASELINE_SNAPS_DIR}")
print(f"Snaps (TL)     : {TIMELAPSE_SNAPS_DIR}")
print(f"Tx scan range  : {tx_positions[0]:.4f} m  ->  {tx_positions[-1]:.4f} m  ({n_traces} traces)")
print(f"Center trace   : #{center_trace}  (Tx at {center_tx_x:.4f} m, Rx at {center_rx_x:.4f} m)")

In [ ]:
# ── B-scan merged .out loader ─────────────────────────────────────────────────
def load_bscan_out(path):
    """Load gprMax B-scan merged HDF5 .out file.

    The merged file has a single receiver (rx1) whose Ez dataset has shape
    (n_iters, n_traces) — time along axis-0, traces along axis-1.
    Returns: traces (n_traces, n_time), time_s (n_time,), dt_s.
    """
    with h5py.File(path, "r") as f:
        dt    = float(f.attrs["dt"])
        iters = int(f.attrs["Iterations"])
        time  = np.arange(iters) * dt
        ez    = np.array(f["rxs/rx1/Ez"])   # (n_iters, n_traces)
    return ez.T, time, dt                   # (n_traces, n_time), time, dt


# ── .vti snapshot loader (identical to visualisation.ipynb) ──────────────────
def load_snapshots(snaps_dir, snap_prefix, indices=None):
    """Load Ez wavefield from .vti snapshot files.
    Returns sorted list of (snap_index, 2D Ez array (ny, nx))."""
    pattern = os.path.join(snaps_dir, f"{snap_prefix}*.vti")
    files   = glob.glob(pattern)
    results = []
    for fpath in files:
        base = os.path.basename(fpath)
        try:
            idx = int(base[len(snap_prefix):-4])
        except ValueError:
            continue
        if indices is not None and idx not in indices:
            continue
        mesh = pv.read(fpath)
        dims = mesh.dimensions
        data = mesh["E-field"]
        Ez   = data[:, 2] if (data.ndim == 2 and data.shape[1] == 3) else data.flatten()
        results.append((idx, Ez.reshape(dims[1] - 1, dims[0] - 1)))
    results.sort(key=lambda x: x[0])
    return results


# ── Load trace data ───────────────────────────────────────────────────────────
print("Loading .out files ...")
baseline_traces,  time, dt = load_bscan_out(BASELINE_OUT)
timelapse_traces, _,    _  = load_bscan_out(TIMELAPSE_OUT)
time_ns = time * 1e9

assert baseline_traces.shape == timelapse_traces.shape, (
    f"Shape mismatch: {baseline_traces.shape} vs {timelapse_traces.shape}"
)
print(f"  Shape  : {baseline_traces.shape}  (n_traces x n_time)")
print(f"  Time   : 0 to {time_ns[-1]:.3f} ns  |  dt = {dt * 1e12:.2f} ps")

# ── Load snapshots (centre trace only) ───────────────────────────────────────
snap_pref_b  = _stem("baseline")  + "_"
snap_pref_tl = _stem("timelapse") + "_"

print("Loading snapshots ...")
snaps_b  = load_snapshots(BASELINE_SNAPS_DIR,  snap_pref_b,  SNAP_INDICES)
snaps_tl = load_snapshots(TIMELAPSE_SNAPS_DIR, snap_pref_tl, SNAP_INDICES)

if snaps_b:
    t0 = snaps_b[0][0]  * snap_interval * 1e9
    t1 = snaps_b[-1][0] * snap_interval * 1e9
    print(f"  Baseline  : {len(snaps_b)} snapshots  (t = {t0:.2f} to {t1:.2f} ns)")
else:
    print("  No baseline snapshots found (run with enable_snapshots=True in BScan_generator)")
if snaps_tl:
    print(f"  Timelapse : {len(snaps_tl)} snapshots")

In [ ]:
# ── B-scan (time-vs-Tx-position heatmap) ─────────────────────────────────────
def plot_bscan(traces, time_ns, tx_pos, title):
    """Seismic colormap B-scan. Colour limits auto-set per call (independent for diff)."""
    vlim = np.percentile(np.abs(traces), 99)
    if vlim == 0:
        vlim = 1.0

    fig, ax = plt.subplots(figsize=(12, 7))
    im = ax.imshow(
        traces.T,
        aspect="auto",
        cmap="seismic",
        origin="upper",
        extent=[tx_pos[0], tx_pos[-1], time_ns[-1], time_ns[0]],
        vmin=-vlim,
        vmax=vlim,
    )

    t_frac_ns = fracture_depth / c_ice * 2.0 * 1e9
    ax.axhline(t_frac_ns, color="lime", linestyle="--", linewidth=1.2,
               label=f"Fracture TWTT ({t_frac_ns:.2f} ns)")

    ax.set_title(title, fontsize=14, weight="bold")
    ax.set_xlabel("Tx position (m)", fontsize=12)
    ax.set_ylabel("Two-way travel time (ns)", fontsize=12)
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.5))
    ax.tick_params(axis="y", which="minor", length=4)
    ax.legend(loc="upper right", fontsize=9)

    cbar = fig.colorbar(im, ax=ax, pad=0.02)
    cbar.set_label("Ez (V/m)", fontsize=10)
    plt.tight_layout()
    return fig


# ── Wiggle / seismic-section plot ─────────────────────────────────────────────
def plot_wiggle(traces, time_ns, tx_pos, title):
    """Wiggle traces. Each trace centred on its Tx x-position."""
    tx_sp   = tx_pos[1] - tx_pos[0] if len(tx_pos) > 1 else trace_spacing
    max_val = np.max(np.abs(traces))
    if max_val == 0:
        max_val = 1.0
    scale = (tx_sp * 0.8) / max_val

    fig, ax = plt.subplots(figsize=(10, 7))
    for i, trace in enumerate(traces):
        x_base = tx_pos[i]
        scaled = x_base + trace * scale
        ax.plot(scaled, time_ns, "k-", linewidth=0.5)
        ax.fill_betweenx(time_ns, x_base, scaled,
                         where=(scaled > x_base), facecolor="k", alpha=0.5)
        ax.axvline(x_base, color="k", linestyle=":", linewidth=0.3, alpha=0.2)

    t_frac_ns = fracture_depth / c_ice * 2.0 * 1e9
    ax.axhline(t_frac_ns, color="lime", linestyle="--", linewidth=1.2,
               label=f"Fracture TWTT ({t_frac_ns:.2f} ns)")

    ax.set_ylim(time_ns[-1], time_ns[0])
    ax.set_xlim(tx_pos[0] - tx_sp, tx_pos[-1] + tx_sp)
    ax.set_title(title, fontsize=14, weight="bold")
    ax.set_xlabel("Tx position (m)", fontsize=12)
    ax.set_ylabel("Two-way travel time (ns)", fontsize=12)
    ax.yaxis.set_minor_locator(ticker.MultipleLocator(0.5))
    ax.tick_params(axis="y", which="minor", length=4)
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    return fig


# ── Snapshot wavefield grid ───────────────────────────────────────────────────
def plot_snapshot_grid(snaps, domain_w, domain_h, snap_dt_s, title, is_diff=False):
    """2-column grid of Ez wavefield panels with geology overlays.
    Markers show the single Tx/Rx pair at the centre-trace position.

    is_diff=True  -> per-panel independent colour scale (residual visualization).
    is_diff=False -> staged normalization: early/mid/late.
    """
    if not snaps:
        print(f"No snapshots to plot: {title}")
        return None

    n      = len(snaps)
    ncols  = 2
    nrows  = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(16, 3.5 * nrows),
                             sharex=True, sharey=True)
    axes = np.array(axes).reshape(-1)

    # ── Staged normalization (mirrors visualisation.ipynb) ────────────────────
    all_abs = [np.abs(d) for _, d in snaps]
    early   = [a for i, a in enumerate(all_abs) if (i / n) <= 0.15]
    mid     = [a for i, a in enumerate(all_abs) if 0.15 < (i / n) <= 0.70]
    late    = [a for i, a in enumerate(all_abs) if (i / n) > 0.70]

    fallback = max(a.max() for a in all_abs) if all_abs else 1.0

    if is_diff:
        global_ref = max((np.percentile(a, 99.5) for a in all_abs), default=fallback)
    else:
        early_max = max((a.max() for a in early), default=fallback)
        mid_pct   = max((np.percentile(a, 99.2) for a in mid),  default=early_max)
        late_pct  = max((np.percentile(a, 99.7) for a in late), default=early_max)
        mid_ref   = float(np.clip(mid_pct,  early_max * 0.035, early_max * 0.22))
        late_ref  = float(np.clip(late_pct, early_max * 0.06,  early_max * 0.32))

    # gprMax y-coordinates (y=0 at bottom, y=domain_h at top)
    y_surface  = domain_h - air_thickness
    y_frac_top = y_surface - fracture_depth

    for i, (idx, data_2d) in enumerate(snaps):
        ax   = axes[i]
        t_ns = idx * snap_dt_s * 1e9
        frac = i / n

        if is_diff:
            vlim = np.percentile(np.abs(data_2d), 99.5) if data_2d.any() else global_ref
        else:
            vlim = early_max if frac <= 0.15 else (mid_ref if frac <= 0.70 else late_ref)
        if vlim == 0:
            vlim = fallback

        im = ax.imshow(
            data_2d,
            aspect="auto",
            cmap="seismic",
            origin="lower",
            extent=[0, domain_w, 0, domain_h],
            vmin=-vlim,
            vmax=vlim,
        )

        # Air layer (light blue patch)
        ax.add_patch(patches.Rectangle(
            (0, y_surface), domain_w, air_thickness,
            facecolor="#cfe8ff", edgecolor="none", alpha=0.30, zorder=3))
        # Ice surface line
        ax.axhline(y_surface,  color="navy", linewidth=0.8, alpha=0.6, zorder=4)
        # Fracture depth line
        ax.axhline(y_frac_top, color="red",  linewidth=0.8,
                   linestyle="--", alpha=0.5, zorder=4)

        # Single Tx star + single Rx triangle at centre-trace position
        ax.plot(center_tx_x, y_surface, "r*", markersize=7, zorder=5,
                label="Tx" if i == 0 else "")
        ax.plot(center_rx_x, y_surface, "gv", markersize=5, zorder=5,
                label="Rx" if i == 0 else "")

        ax.set_title(f"t = {t_ns:.2f} ns", fontsize=10, weight="bold")
        ax.set_xlabel("Length (m)", fontsize=9)
        ax.set_ylabel("Depth (m)",  fontsize=9)

        if i == 0:
            ax.legend(loc="lower right", fontsize=8, ncol=2)

        cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
        cbar.set_label("Ez (V/m)", fontsize=8)
        cbar.ax.tick_params(labelsize=7)

    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.suptitle(title, fontsize=14, weight="bold")
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    return fig

In [ ]:
# ── Difference traces ─────────────────────────────────────────────────────────
diff_traces         = baseline_traces - timelapse_traces
diff_traces_display = diff_traces * DIFF_GAIN

ratio = np.max(np.abs(diff_traces)) / (np.max(np.abs(baseline_traces)) + 1e-30)
print(f"max|baseline|  = {np.max(np.abs(baseline_traces)):.3e}")
print(f"max|diff|      = {np.max(np.abs(diff_traces)):.3e}  (ratio {ratio:.2e})")
print(f"Display gain   = x{DIFF_GAIN}  =>  displayed max = {np.max(np.abs(diff_traces_display)):.3e}")

# ── Difference snapshots ──────────────────────────────────────────────────────
if snaps_b and snaps_tl:
    snap_pairs = [
        (idx_b, d_b - d_tl)
        for (idx_b, d_b), (idx_tl, d_tl) in zip(snaps_b, snaps_tl)
        if idx_b == idx_tl
    ]
    if len(snap_pairs) < len(snaps_b):
        print(f"Warning: {len(snaps_b) - len(snap_pairs)} snapshot indices did not align "
              f"-- verify snap_interval matches BScan_generator.")
    snaps_diff_display = [(idx, data * DIFF_GAIN) for idx, data in snap_pairs]
else:
    snap_pairs = []
    snaps_diff_display = []

# Domain height derived from snapshot shape (avoids hard-coding fracture_aperture)
if snaps_b:
    _, _s = snaps_b[0]
    snap_domain_h = _s.shape[0] / _s.shape[1] * domain_width
else:
    snap_domain_h = air_thickness + fracture_depth + 0.2

# ─────────────────────────────────────────────────────────────────────────────
# 1. B-scan (time-vs-Tx-position heatmap)
# ─────────────────────────────────────────────────────────────────────────────
print("\n=== B-scans ===")
fig_b1 = plot_bscan(baseline_traces,     time_ns, tx_positions, "Baseline — B-scan")
plt.show()

fig_b2 = plot_bscan(timelapse_traces,    time_ns, tx_positions, "Timelapse — B-scan")
plt.show()

fig_b3 = plot_bscan(diff_traces_display, time_ns, tx_positions,
                    f"Difference ×{DIFF_GAIN:.0f} — B-scan  (independent colour scale)")
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# 2. Wiggle / seismic-section
# ─────────────────────────────────────────────────────────────────────────────
print("\n=== Wiggle plots ===")
fig_w1 = plot_wiggle(baseline_traces,     time_ns, tx_positions, "Baseline — Wiggle")
plt.show()

fig_w2 = plot_wiggle(timelapse_traces,    time_ns, tx_positions, "Timelapse — Wiggle")
plt.show()

fig_w3 = plot_wiggle(diff_traces_display, time_ns, tx_positions,
                     f"Difference ×{DIFF_GAIN:.0f} — Wiggle")
plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# 3. Wavefield snapshot grids (centre trace only)
# ─────────────────────────────────────────────────────────────────────────────
if snaps_b or snaps_tl or snaps_diff_display:
    print("\n=== Wavefield snapshots (centre trace) ===")

if snaps_b:
    fig_s1 = plot_snapshot_grid(
        snaps_b, domain_width, snap_domain_h, snap_interval,
        "Baseline — Wavefield Snapshots  (centre trace)")
    plt.show()

if snaps_tl:
    fig_s2 = plot_snapshot_grid(
        snaps_tl, domain_width, snap_domain_h, snap_interval,
        "Timelapse — Wavefield Snapshots  (centre trace)")
    plt.show()

if snaps_diff_display:
    fig_s3 = plot_snapshot_grid(
        snaps_diff_display, domain_width, snap_domain_h, snap_interval,
        f"Difference ×{DIFF_GAIN:.0f} — Wavefield Snapshots  (per-panel colour scale)",
        is_diff=True)
    plt.show()

## Kirchhoff Migration

Applies **common-offset Kirchhoff migration** to the baseline, timelapse, and difference
B-scans using [`pylops.waveeqprocessing.Kirchhoff`](https://pylops.readthedocs.io).

### Why a per-trace adjoint loop?

Standard Kirchhoff PSTM assumes a **Common Shot Gather**: one fixed source, N receivers.
A B-scan has **common-offset** geometry — Tx and Rx move together, so every trace has its
own unique source-receiver pair. A single Pylops operator cannot express this without
creating an N×N source-receiver matrix (N² pairs, most empty).

The solution: build one `Kirchhoff(nsrc=1, nrec=1)` operator per trace and accumulate the
adjoint (migration) images.

**Traveltime at image point (x, z) for trace i:**

$$t_i(x,z) = \frac{\sqrt{(x - x_{\mathrm{Tx},i})^2 + z^2} + \sqrt{(x - x_{\mathrm{Rx},i})^2 + z^2}}{v}$$

Since offset (2 cm) ≪ λ_ice (11.3 cm), this is very close to the zero-offset formula.

In [ ]:
import os as _os, time as _time
_os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
from pylops.waveeqprocessing import Kirchhoff

# ── Migration image grid ───────────────────────────────────────────────────────
z_img = np.linspace(0.0, 1.2, 120)      # depth axis   [m]
x_img = np.linspace(0.0, domain_width, 200)  # lateral axis [m]

rx_positions = tx_positions + offset    # per-trace receiver x-positions

# ── Ricker wavelet matched to the simulation ───────────────────────────────────
time_s = time_ns * 1e-9
# dt is already in seconds from load_bscan_out

def _make_ricker(dt_s, f_hz, nt_wav=501):
    t   = np.arange(nt_wav) * dt_s
    t0  = t[nt_wav // 2]
    tau = t - t0
    u   = (np.pi * f_hz * tau) ** 2
    return (1.0 - 2.0 * u) * np.exp(-u), nt_wav // 2

wav, wavcenter = _make_ricker(dt, f_central)
print(f"Wavelet: {len(wav)} samples, centre at index {wavcenter}")
print(f"Image grid: {len(z_img)} depth × {len(x_img)} lateral points")


# ── Common-offset Kirchhoff migration ─────────────────────────────────────────
def kirchhoff_bscan(bscan, time_s, tx_pos, rx_pos, v_ms, x_img, z_img,
                    wav, wavcenter, angleaperture=60.0):
    """Migrate a common-offset B-scan using Pylops Kirchhoff adjoint.

    Builds one Kirchhoff(nsrc=1, nrec=1) per trace and stacks adjoint images.
    Peak-normalises the final image.
    Returns: image (nz, nx).
    """
    n_traces = bscan.shape[0]
    nz, nx   = len(z_img), len(x_img)
    image    = np.zeros((nz, nx))

    t0 = _time.perf_counter()
    for i in range(n_traces):
        src_i = np.array([[tx_pos[i]], [0.0]])   # shape (2,1): row0=x, row1=z
        rec_i = np.array([[rx_pos[i]], [0.0]])
        Kop   = Kirchhoff(
            z_img, x_img, time_s,
            src_i, rec_i, v_ms,
            wav, wavcenter,
            mode='analytic',
            angleaperture=angleaperture,
            dynamic=False,
        )
        image += (Kop.H @ bscan[i, :]).reshape(nz, nx)

    elapsed = _time.perf_counter() - t0
    print(f"  {n_traces} traces in {elapsed:.1f} s  ({elapsed / n_traces * 1e3:.1f} ms/trace)")
    peak = np.max(np.abs(image))
    return image / peak if peak > 0 else image

In [ ]:
v_ice_ms = c_ice   # [m/s]

print("=== Pylops Kirchhoff Migration ===")
datasets = {
    "Baseline":   baseline_traces,
    "Timelapse":  timelapse_traces,
    "Difference": diff_traces,
}
migrated = {}
for name, bs in datasets.items():
    print(f"  [{name}] ...")
    migrated[name] = kirchhoff_bscan(
        bs, time_s, tx_positions, rx_positions, v_ice_ms,
        x_img, z_img, wav, wavcenter,
    )
    print(f"  [{name}] done")

# ── 2×3 overview: raw B-scans (top) + migrated images (bottom) ───────────────
labels = ["Baseline", "Timelapse", "Difference"]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, label in enumerate(labels):
    raw = (baseline_traces  if label == "Baseline"  else
           timelapse_traces if label == "Timelapse" else diff_traces_display)

    vlim = np.percentile(np.abs(raw), 99)
    axes[0, col].imshow(
        raw.T, aspect="auto", cmap="seismic", origin="upper",
        extent=[tx_positions[0], tx_positions[-1], time_ns[-1], time_ns[0]],
        vmin=-vlim, vmax=vlim,
    )
    axes[0, col].set_title(f"{label} — Raw B-scan", fontsize=11, weight="bold")
    axes[0, col].set_xlabel("Tx position (m)", fontsize=9)
    axes[0, col].set_ylabel("TWTT (ns)", fontsize=9)

    img    = migrated[label]
    vlim_m = np.percentile(np.abs(img), 99)
    im = axes[1, col].imshow(
        img, aspect="auto", cmap="seismic", origin="upper",
        extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]],
        vmin=-vlim_m, vmax=vlim_m,
    )
    axes[1, col].axhline(fracture_depth, color="lime", linestyle="--",
                         linewidth=1.2, label=f"Fracture ({fracture_depth} m)")
    axes[1, col].set_title(f"{label} — Kirchhoff Migrated", fontsize=11, weight="bold")
    axes[1, col].set_xlabel("x (m)", fontsize=9)
    axes[1, col].set_ylabel("Depth (m)", fontsize=9)
    if col == 0:
        axes[1, col].legend(fontsize=8)
    fig.colorbar(im, ax=axes[1, col], fraction=0.035, pad=0.02).set_label(
        "Normalised amplitude", fontsize=8)

fig.suptitle("Pylops Kirchhoff Migration — Overview", fontsize=14, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# ── Zoomed difference image around the fracture ───────────────────────────────
fig2, ax2 = plt.subplots(figsize=(12, 5))
img_diff = migrated["Difference"]
vlim_d   = np.percentile(np.abs(img_diff), 99.5)
im2 = ax2.imshow(
    img_diff, aspect="auto", cmap="seismic", origin="upper",
    extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]],
    vmin=-vlim_d, vmax=vlim_d,
)
ax2.set_ylim(fracture_depth + 0.2, max(0.0, fracture_depth - 0.2))
ax2.axhline(fracture_depth, color="lime", linestyle="--", linewidth=1.5,
            label=f"Fracture depth ({fracture_depth} m)")
ax2.set_title(
    "Difference — Kirchhoff Migrated  (zoomed to fracture zone)",
    fontsize=13, weight="bold",
)
ax2.set_xlabel("x (m)", fontsize=11)
ax2.set_ylabel("Depth (m)", fontsize=11)
ax2.legend(fontsize=10)
fig2.colorbar(im2, ax=ax2, pad=0.02).set_label("Normalised amplitude", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
A = migrated["Baseline"]
B = migrated["Timelapse"]
diff = A - B
plt.imshow(diff, aspect="auto", cmap="seismic", origin="upper",
           extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]])
plt.colorbar(label="Difference amplitude")
plt.title("Difference between Baseline and Timelapse Migrated Images")
plt.xlabel("x (m)")
plt.ylabel("Depth (m)")
plt.show()



## Phase-Shift (Gazdag) Migration

### Why Gazdag works better here than Kirchhoff (adjoint)

| | Kirchhoff `.H` | Gazdag Phase-Shift |
|---|---|---|
| Method | Ray-based backprojection | Wave-equation, full angular spectrum |
| Flat horizontal reflectors | Spreads energy along elliptic arcs → vertical stripe artefacts | Images correctly at the right depth |
| Diffractions | Collapses hyperbolas | Also collapses hyperbolas |
| Cost | O(N\_traces × N\_z × N\_x) — slow | O(N\_z × N\_x log N\_x) — fast |

### Algorithm for common-offset B-scan (exploding-reflector model)

Since the Tx–Rx offset (2 cm) ≪ λ\_ice (11 cm) and ≪ fracture depth (60 cm), the data is treated as **zero-offset** using the **exploding-reflector model**: reflectors "explode" at t = 0 and the wavefield propagates upward at **v / 2**, so the one-way phase shift accounts for the full two-way travel time.

No source Green's function correction is needed (unlike the CSG version in Migration\_Playground.ipynb) because Tx and Rx move together and the standard downward continuation handles both legs symmetrically.

**Per-depth step Δz:**

$$D(k_x,\,\omega,\,z+\Delta z) = D(k_x,\,\omega,\,z)\; e^{+i k_z \Delta z}, \qquad k_z = \sqrt{\!\left(\frac{2\omega}{v}\right)^{\!2} - k_x^2}$$

**Imaging condition** — extract the t = 0 value by summing over all frequencies:

$$I(x, z) = \operatorname{Re}\!\left\{\mathcal{F}^{-1}_{k_x}\!\left\{\sum_\omega D(k_x,\omega,z)\right\}\right\}$$

Evanescent components ($k_x^2 > (2\omega/v)^2$) are zeroed before the loop.  
A Hanning spatial taper and 2× zero-padding suppress scan-edge ringing and periodic aliases.

In [ ]:
import time as _time_mod
from scipy.interpolate import interp1d as _interp1d

# ── Migration velocity & image grid ───────────────────────────────────────────
# z_img / x_img defined in the Kirchhoff cell above; reuse them here.
v_ice_mns = c_ice * 1e-9    # wave speed in ice [m/ns]

# ── Optional direct-wave mute ─────────────────────────────────────────────────
# Set > 0 to zero the first t_mute_ns of each trace before migration.
# Useful if strong direct-wave energy dominates the image near z = 0.
# Rule of thumb: mute up to just before the first real reflection.
t_mute_ns = 6.0             # [ns]  0 = no mute


# ── Gazdag Phase-Shift migration for common-offset B-scan ────────────────────
def phase_shift_migration_bscan(bscan, time_ns, tx_pos, v_mns, x_img, z_img,
                                 t_mute_ns=0.0):
    """Zero-offset Gazdag migration for a common-offset GPR B-scan.

    Treats the data as zero-offset (valid when offset << lambda).
    Uses the exploding-reflector model: v_mig = v/2 so the one-way
    phase shift accounts for the two-way travel time on the TWTT axis.

    bscan     : (n_traces, n_time)
    tx_pos    : (n_traces,)  Tx x-positions [m]
    v_mns     : medium velocity [m/ns]
    t_mute_ns : zero first t_mute_ns of every trace (removes direct wave)

    Returns   : image (nz, nx), peak-normalised.
    """
    n_tr, n_t = bscan.shape
    dt_ns = time_ns[1] - time_ns[0]     # [ns]
    dx    = tx_pos[1] - tx_pos[0]       # trace spacing [m]
    v_half = v_mns / 2.0                # exploding-reflector velocity [m/ns]

    data = bscan.copy()
    if t_mute_ns > 0.0:
        mute_samp = int(t_mute_ns / dt_ns)
        data[:, :mute_samp] = 0.0

    # 2× zero-padding in x: pushes periodic alias images outside [0, 2*scan_width]
    nfft_x = 2 * n_tr

    # ── Frequency / wavenumber grids ──────────────────────────────────────────
    omega = 2.0 * np.pi * np.fft.fftfreq(n_t,    d=dt_ns)   # [rad/ns]  (n_t,)
    kx    = 2.0 * np.pi * np.fft.fftfreq(nfft_x, d=dx)      # [rad/m] (nfft_x,)
    KX, OM = np.meshgrid(kx, omega, indexing='ij')            # (nfft_x, n_t)

    # ── Dispersion relation (one-way at v/2) ──────────────────────────────────
    arg = (OM / v_half) ** 2 - KX ** 2
    KZ  = np.where(arg > 0.0, np.sqrt(arg.clip(0)), 0.0)     # [rad/m]

    # ── Spatial taper → reduce Gibbs ringing at scan edges ───────────────────
    taper = np.hanning(n_tr)

    # ── 2D FFT with spatial zero-padding → D(kx, ω) ──────────────────────────
    D_accum = np.fft.fft2(data * taper[:, np.newaxis], s=(nfft_x, n_t))
    D_accum[arg <= 0.0] = 0.0          # zero evanescent components before loop

    # x-axis of the IFFT output (starts at tx_pos[0], spacing dx)
    x_ifft = tx_pos[0] + np.arange(nfft_x) * dx

    image  = np.zeros((len(z_img), nfft_x))
    z_prev = 0.0
    t0     = _time_mod.perf_counter()

    for iz, z_out in enumerate(z_img):
        dz = z_out - z_prev
        if dz > 0.0:
            D_accum *= np.exp(1j * KZ * dz)           # phase-shift by dz

        # Imaging condition: IFFT along kx → x, then sum over ω (= t=0 value)
        D_x_w     = np.fft.ifft(D_accum, axis=0)      # (nfft_x, n_t)
        image[iz] = np.real(np.sum(D_x_w, axis=1))    # sum freq → Re{d(x, t=0)}
        z_prev = z_out

    elapsed = _time_mod.perf_counter() - t0
    print(f"  {elapsed:.1f} s  ({elapsed / len(z_img) * 1e3:.1f} ms/depth)")

    # ── Interpolate padded grid → x_img ──────────────────────────────────────
    image_out = np.zeros((len(z_img), len(x_img)))
    for iz in range(len(z_img)):
        image_out[iz] = _interp1d(
            x_ifft, image[iz], kind='linear',
            bounds_error=False, fill_value=0.0,
        )(x_img)

    peak = np.max(np.abs(image_out))
    return image_out / peak if peak > 0 else image_out


print(f"v_ice_mns = {v_ice_mns:.4f} m/ns  |  v/2 = {v_ice_mns/2:.4f} m/ns")
print(f"Image grid : {len(z_img)} depth × {len(x_img)} lateral")

In [ ]:
n_actual = baseline_traces.shape[0]
tx_pos_actual = tx_positions[:n_actual]   # guard for off-by-one if n_traces > data rows

print("=== Phase-Shift (Gazdag) Migration ===")
ps_datasets = {
    "Baseline":   baseline_traces,
    "Timelapse":  timelapse_traces,
    "Difference": diff_traces,
}
ps_migrated = {}
for name, bs in ps_datasets.items():
    print(f"  [{name}] ...")
    ps_migrated[name] = phase_shift_migration_bscan(
        bs, time_ns, tx_pos_actual, v_ice_mns, x_img, z_img,
        t_mute_ns=t_mute_ns,
    )
    print(f"  [{name}] done")

# ── 2×3 overview: raw B-scans (top) + Gazdag images (bottom) ─────────────────
labels = ["Baseline", "Timelapse", "Difference"]
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, label in enumerate(labels):
    raw = (baseline_traces  if label == "Baseline"  else
           timelapse_traces if label == "Timelapse" else diff_traces_display)

    vlim = np.percentile(np.abs(raw), 99)
    axes[0, col].imshow(
        raw.T, aspect="auto", cmap="seismic", origin="upper",
        extent=[tx_positions[0], tx_positions[-1], time_ns[-1], time_ns[0]],
        vmin=-vlim, vmax=vlim,
    )
    axes[0, col].set_title(f"{label} — Raw B-scan", fontsize=11, weight="bold")
    axes[0, col].set_xlabel("Tx position (m)", fontsize=9)
    axes[0, col].set_ylabel("TWTT (ns)", fontsize=9)

    img    = ps_migrated[label]
    vlim_m = np.percentile(np.abs(img), 99)
    im = axes[1, col].imshow(
        img, aspect="auto", cmap="seismic", origin="upper",
        extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]],
        vmin=-vlim_m, vmax=vlim_m,
    )
    axes[1, col].axhline(fracture_depth, color="lime", linestyle="--",
                         linewidth=1.2, label=f"Fracture ({fracture_depth} m)")
    # Air-ice interface reference
    axes[1, col].axhline(air_thickness, color="cyan", linestyle=":",
                         linewidth=1.0, label=f"Ice surface ({air_thickness} m)")
    axes[1, col].set_title(f"{label} — Gazdag Migrated", fontsize=11, weight="bold")
    axes[1, col].set_xlabel("x (m)", fontsize=9)
    axes[1, col].set_ylabel("Depth (m)", fontsize=9)
    if col == 0:
        axes[1, col].legend(fontsize=8)
    fig.colorbar(im, ax=axes[1, col], fraction=0.035, pad=0.02).set_label(
        "Normalised amplitude", fontsize=8)

fig.suptitle("Gazdag Phase-Shift Migration — Overview", fontsize=14, weight="bold")
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# ── Zoomed difference image around the fracture ───────────────────────────────
fig2, ax2 = plt.subplots(figsize=(12, 5))
img_diff_ps = ps_migrated["Difference"]
vlim_d      = np.percentile(np.abs(img_diff_ps), 99.5)
im2 = ax2.imshow(
    img_diff_ps, aspect="auto", cmap="seismic", origin="upper",
    extent=[x_img[0], x_img[-1], z_img[-1], z_img[0]],
    vmin=-vlim_d, vmax=vlim_d,
)
ax2.set_ylim(fracture_depth + 0.25, max(0.0, fracture_depth - 0.25))
ax2.axhline(fracture_depth, color="lime",  linestyle="--", linewidth=1.5,
            label=f"Fracture depth ({fracture_depth} m)")
ax2.axhline(air_thickness,  color="cyan",  linestyle=":",  linewidth=1.0,
            label=f"Ice surface ({air_thickness} m)")
ax2.set_title(
    "Difference — Gazdag Migrated  (zoomed to fracture zone)",
    fontsize=13, weight="bold",
)
ax2.set_xlabel("x (m)", fontsize=11)
ax2.set_ylabel("Depth (m)", fontsize=11)
ax2.legend(fontsize=10)
fig2.colorbar(im2, ax=ax2, pad=0.02).set_label("Normalised amplitude", fontsize=10)
plt.tight_layout()
plt.show()